In [2]:
#@title 1. เตรียมระบบและโหลดโมเดล 4x-UltraSharp (กดรันครั้งเดียว)
import os
import torch

# 1. ติดตั้งไลบรารีที่จำเป็นโดยไม่ฝืนอัปเกรดตัวหลักจนเวอร์ชันตีกัน
print("กำลังติดตั้ง Library... (ใช้รุ่นเสถียรสำหรับระบบ)")
!pip install -q spandrel gradio opencv-python-headless pillow

# 2. เคลียร์ไฟล์เก่าถ้าหากขนาดไม่ถูกต้องหรือเสียหาย
file_path = '4x-UltraSharp.pth'
if os.path.exists(file_path):
    if os.path.getsize(file_path) < 1024 * 1024:  # น้อยกว่า 1MB คือไฟล์ HTML แน่ๆ
        print("พบไฟล์เก่าที่ชำรุด กำลังลบเพื่อเตรียมดาวน์โหลดใหม่...")
        os.remove(file_path)

# 3. ดาวน์โหลดโมเดลจากลิงก์ที่ถูกต้อง
if not os.path.exists(file_path):
    print("กำลังดาวน์โหลดโมเดล 4x-UltraSharp.pth (ขนาดประมาณ 64MB)...")
    !wget -q --show-progress https://huggingface.co/lokcx/4x-Ultrasharp/resolve/main/4x-UltraSharp.pth -O 4x-UltraSharp.pth

# 4. โหลดโมเดลเข้าระบบเพื่อตรวจสอบความถูกต้อง (รองรับ CPU fallback)
print("กำลังโหลดโมเดลเข้าสู่หน่วยความจำ...")
from spandrel import ModelLoader

try:
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print("พบ GPU! ระบบจะประมวลผลด้วยฮาร์ดแวร์ GPU (ความเร็วสูง)")
    else:
        device = torch.device('cpu')
        # ตั้งค่าจำนวน thread สำหรับการทำงานบน CPU เพื่อความรวดเร็วสูงสุด
        torch.set_num_threads(os.cpu_count() or 4)
        print(f"⚠️ ไม่พบ GPU ระบบสลับไปใช้ CPU ในการทำงานอัตโนมัติ (จะช้ากว่าปกติเล็กน้อย) ใช้ CPU Threads: {torch.get_num_threads()}")

    model = ModelLoader().load_from_file('4x-UltraSharp.pth')
    model.to(device).eval()
    print(f'\n✅ โหลดโมเดล 4x-UltraSharp บน {device} สำเร็จและพร้อมใช้งาน!')
except Exception as e:
    print(f'\n❌ โหลดโมเดลไม่สำเร็จ: {e}')
    print("แนะนำให้รันเซลล์นี้ใหม่อีกครั้งเพื่อทำการโหลดซ่อมแซมครับ")

กำลังติดตั้ง Library... (ใช้รุ่นเสถียรสำหรับระบบ)
กำลังโหลดโมเดลเข้าสู่หน่วยความจำ...
พบ GPU! ระบบจะประมวลผลด้วยฮาร์ดแวร์ GPU (ความเร็วสูง)

✅ โหลดโมเดล 4x-UltraSharp บน cuda สำเร็จและพร้อมใช้งาน!


In [5]:
import gradio as gr
import cv2
import numpy as np
import os
import shutil
import time
import threading
import queue
from collections import deque
import torch

os.makedirs('input_queue', exist_ok=True)
os.makedirs('output_done', exist_ok=True)

task_queue = queue.Queue()
pending_names = deque()
current_task = 'ว่าง (พร้อมรับรูป)'
completed_files = []
lock = threading.Lock()

# เก็บข้อมูลประวัติการทำงานและระยะเวลาที่ใช้
processed_history = []

def upscale_tiled(model, tensor, tile_size=512, overlap=32):
    b, c, h, w = tensor.shape
    scale = model.scale
    output = torch.zeros((b, c, h * scale, w * scale), device=tensor.device)
    weight = torch.zeros_like(output)
    for y in range(0, h, tile_size - overlap):
        for x in range(0, w, tile_size - overlap):
            y_end = min(y + tile_size, h)
            x_end = min(x + tile_size, w)
            y_start = max(0, y_end - tile_size)
            x_start = max(0, x_end - tile_size)
            tile = tensor[:, :, y_start:y_end, x_start:x_end]
            with torch.no_grad():
                out_tile = model(tile)
            output[:, :, y_start*scale:y_end*scale, x_start*scale:x_end*scale] += out_tile
            weight[:, :, y_start*scale:y_end*scale, x_start*scale:x_end*scale] += 1.0
    return output / weight

def worker():
    global current_task
    while True:
        filepath, orig_name = task_queue.get()
        start_time = time.time()
        with lock:
            if orig_name in pending_names:
                pending_names.remove(orig_name)
            current_task = f'กำลังขยายภาพ: {orig_name}'
        try:
            img = cv2.imread(filepath, cv2.IMREAD_COLOR)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img_tensor = torch.from_numpy(img).permute(2, 0, 1).float().div(255.0).unsqueeze(0).to(device)
                with torch.no_grad():
                    out_tensor = upscale_tiled(model, img_tensor, tile_size=512, overlap=32)
                out_img = out_tensor.squeeze(0).permute(1, 2, 0).clamp(0, 1).mul(255).byte().cpu().numpy()
                out_img = cv2.cvtColor(out_img, cv2.COLOR_RGB2BGR)
                out_name = f'4x_{os.path.splitext(orig_name)[0]}.png'
                out_path = os.path.join('output_done', out_name)
                cv2.imwrite(out_path, out_img)

                elapsed_time = time.time() - start_time
                with lock:
                    completed_files.append(out_path)
                    # บันทึกประวัติและระยะเวลาที่ใช้งานจริงของแต่ละภาพ
                    processed_history.append(f"{out_name} (ใช้เวลา: {elapsed_time:.2f} วินาที)")
        except Exception as e:
            print(f'Error processing {orig_name}: {e}')
        finally:
            with lock:
                current_task = 'ว่าง (พร้อมรับรูป)'
            task_queue.task_done()

# รัน Worker Thread เพียงครั้งเดียวในเบื้องหลัง
try:
    # ค้นหาว่ามี thread ที่เคยรันไว้รึยังเพื่อไม่ให้สร้างทับซ้อนกันจนแย่งทรัพยากร
    has_worker = False
    for t in threading.enumerate():
        if t.name == "UpscaleWorker":
            has_worker = True
            break
    if not has_worker:
        t = threading.Thread(target=worker, name="UpscaleWorker", daemon=True)
        t.start()
except Exception as e:
    pass

def add_files(files):
    if not files:
        return None, get_status(), gr.update(), get_history_text()
    for f in files:
        src = f.name if hasattr(f, 'name') else f
        orig_name = os.path.basename(src)
        dst = os.path.join('input_queue', f'{int(time.time()*1000)}_{orig_name}')
        shutil.copy(src, dst)
        with lock:
            pending_names.append(orig_name)
        task_queue.put((dst, orig_name))
    return None, get_status(), gr.update(value=completed_files), get_history_text()

def get_status():
    with lock:
        q_list = list(pending_names)
        status = f'ระบบรันบนอุปกรณ์: {device.type.upper()}\n'
        status += f'สถานะปัจจุบัน: {current_task}\n'
        status += f'รอในคิว ({len(q_list)} ภาพ): ' + (', '.join(q_list) if q_list else 'ไม่มี')
    return status

def get_history_text():
    with lock:
        if not processed_history:
            return "ยังไม่มีประวัติการประมวลผล"
        return "\n".join(processed_history[::-1]) # แสดงอันล่าสุดขึ้นก่อน

def refresh_ui():
    with lock:
        files_copy = list(completed_files)
    return get_status(), files_copy, get_history_text()

auto_download_js = '''
() => {
    window._downloadedSet = window._downloadedSet || new Set();
    setInterval(() => {
        const downloadElements = document.querySelectorAll("#done_files a");
        downloadElements.forEach(el => {
            const href = el.getAttribute("href");
            if (href && !window._downloadedSet.has(href)) {
                window._downloadedSet.add(href);

                const link = document.createElement("a");
                link.href = href;
                link.download = href.split("/").pop() || "image.png";
                document.body.appendChild(link);
                link.click();
                document.body.removeChild(link);
            }
        });
    }, 1000);
}
'''

with gr.Blocks(title="4x Auto-Queue Upscaler") as demo:
    gr.Markdown('## ⚡ 4x Auto-Queue Upscaler')
    gr.Markdown('*คำแนะนำ: คลิกขวาที่ลิงก์ด้านล่าง `.gradio.live` แล้วเปิดในแท็บใหม่ เพื่อการดาวน์โหลดอัตโนมัติที่ลื่นไหลที่สุดและไม่ติด Sandbox บล็อก*')

    with gr.Row():
        # คอลัมน์ซ้าย: จัดการอินพุตและสถานะการประมวลผล
        with gr.Column(scale=1):
            file_in = gr.File(label='1. เลือกภาพหรือลากมาวางเพื่อเข้าคิว (ปรับขนาดปุ่มให้เล็กลง)', file_count='multiple', file_types=['image'], height=150)
            status_box = gr.Textbox(label='2. สถานะการทำงานของคิวปัจจุบัน', value=get_status(), lines=3, interactive=False)
            history_box = gr.Textbox(label='⏱️ บันทึกเวลาที่ใช้แต่ละภาพ (ล่าสุดอยู่บนสุด)', value=get_history_text(), lines=6, interactive=False)

        # คอลัมน์ขวา: แสดงผลลัพธ์เพื่อการดาวน์โหลด
        with gr.Column(scale=1):
            file_out = gr.File(label='3. ไฟล์ที่อัปสเกลเสร็จแล้ว (จะดาวน์โหลดอัตโนมัติหากเปิดในแท็บใหม่)', elem_id='done_files', interactive=False, height=360)

    file_in.upload(fn=add_files, inputs=[file_in], outputs=[file_in, status_box, file_out, history_box])

    timer = gr.Timer(2)
    timer.tick(fn=refresh_ui, outputs=[status_box, file_out, history_box])

demo.queue().launch(share=True, inline=True, js=auto_download_js)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ec763797321bfedeb4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
